## 1. Setup and data

### 1.1 Packages and imports

The first cell checks that the packages beyond the usual data-science set are installed (tqdm, CatBoost, Hyperopt and scikit-learn), installs any that are missing and shows a progress bar while it does. Then it imports everything this notebook uses.

All packages are also listed in `requirements.txt`, so after `pip install -r requirements.txt` the check simply confirms that everything is there.

In [ ]:
import importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
import subprocess
import sys
import time

RUN_ALL_START = time.perf_counter() ## Notebook runtime

# Package verification and installation
packages_to_check = {
    "tqdm": "tqdm",
    "gdown": "gdown",
    "catboost": "catboost",
    "scikit-learn": "sklearn",
    "hyperopt": "hyperopt",
    "joblib": "joblib",
}

missing_packages = [
    install_name
    for install_name, import_name in packages_to_check.items()
    if importlib.util.find_spec(import_name) is None
]

progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Setup:",
    bar_style="info",
)

status = widgets.HTML(value="Starting setup...")
display(progress, status)

progress.value = 20
status.value = "Checking required packages..."

if missing_packages:
    step_value = 70 / len(missing_packages)

    for index, package in enumerate(missing_packages, start=1):
        status.value = f"Installing package: {package}..."

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", package],
            check=True,
        )

        progress.value = int(20 + index * step_value)

progress.value = 100
progress.bar_style = "success"
status.value = "✓ All required packages are available."

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from tqdm.notebook import tqdm

from catboost import CatBoostClassifier
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll.base import scope

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### 1.2 Settings used throughout

This cell defines the shared random state, cross-validation folds, test-set size, scoring metric, and number of Hyperopt trials.

All models use the same stratified 5-fold split, so their macro F1 scores are directly comparable. Stratification preserves the class proportions in every fold.

The colour settings ensure a consistent style across plots and output boxes.

In [ ]:
## 1.2 Settings
RANDOM_STATE = 42       # the same number everywhere, so results are reproducible
N_SPLITS = 5            # 5-fold cross-validation
TEST_SIZE = 0.2         # 20% of the data is held back and never used for tuning
SCORING = "f1_macro"    # our comparison metric — see section 3 for why
MAX_EVALS = 10          # how many settings Hyperopt tries per model

# StratifiedKFold keeps the class proportions the same in every fold.
# That matters here because our classes are imbalanced.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# --- The five OCEAN traits (column names start with their letter) ---------
TRAITS = {
    "O": "Openness to Experience",
    "C": "Conscientiousness",
    "E": "Extraversion",
    "A": "Agreeableness",
    "N": "Neuroticism"
}

ocean_order = list(TRAITS.values())

# --- One fixed colour per personality type, used in every plot -------------
TARGET_COLORS = {
    "Resilient":        "#e07a5f",  # muted terracotta/coral
    "Overcontroller":   "#81b29a",  # sage/mint green
    "Undercontroller":  "#f2cc8f",  # warm ochre yellow
    "Moderate":         "#2b5c6f"   # petroleum blue (same as the primary theme colour)
}

target_order = list(TARGET_COLORS.keys())

# --- Colours for titles, info boxes and heatmaps -----------------------------
THEME = {
    "primary":          "#2b5c6f",  # Main petroleum blue for titles & headers
    "success_bg":       "#edf7f4",  # Soft mint/sage background
    "success_border":   "#81b29a",  # Sage/mint green border (matches Overcontroller)
    "danger_bg":        "#fdf2f2",  # Soft coral/red background
    "danger_border":    "#e07a5f",  # Terracotta/coral border (matches Resilient)
    "palette":          "Blues"
}

### 1.3 Load the cleaned data and hold out a test set

The group's EDA notebook (`eda.ipynb`) cleans the raw data and saves the result as `data/data_clean.csv`. The cleaning steps, and how many rows each one removed, are documented there.

> The `data/` folder is **not** in the repository. Download the dataset with the Drive link in the README, or simply run `eda.ipynb`: it downloads the raw file itself, creates the `data/` folder and saves the cleaned file there. If the cleaned file is missing, the cell below stops with a message to run `eda.ipynb` first.

`X` holds the 22 input columns (19 statements, `age`, `gender`, `hand`) and `y` the personality type. We then split off **20% as a test set**. The test set is never used to train, tune or choose a model. Everything that decides something (cross-validation, grid search, Hyperopt, picking the champion) happens on the training 80% only.

| Set | Used for |
|---|---|
| **Train (80%)** | Cross-validation, model comparison, hyperparameter tuning |
| **Test (20%)** | A final check on data the models have never seen |

`stratify=y` keeps the same class proportions in both parts.

In [ ]:
# 1.3 Load data
DATA_FILE = Path("data") / "data_clean.csv"
df = pd.read_csv(DATA_FILE)

X = df.drop(columns="target")   # features: 19 answers + age + gender + hand
y = df["target"]                # what we predict, kept as readable text

# the 19 questionnaire columns = everything except the demographics and the target
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,                 # same class mix in both halves
)

display(HTML(f"""
<table style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']};
              border-collapse:collapse; margin:8px 0; font-size:14px; text-align:left;">
    <tr>
        <td colspan="2" style="padding:10px 14px; border:none; text-align:left;">
            <b>Total:</b> {len(df):,} rows
        </td>
    </tr>
    <tr>
        <td style="padding:4px 14px; border:none; text-align:left;">
            <b>Train:</b> {len(X_train):,} rows
        </td>
        <td style="padding:4px 14px; border:none; text-align:left;">
            <b>Test:</b> {len(X_test):,} rows
        </td>
    </tr>
    <tr>
        <td style="padding:4px 14px 10px; border:none; text-align:left;">
            <b>Features:</b> {X.shape[1]}
        </td>
        <td style="padding:4px 14px 10px; border:none; text-align:left;">
            <b>Items:</b> {len(item_cols)}
        </td>
    </tr>
</table>
"""))

df.head()

In [ ]:
# 1.4 How many people of each personality type?
# The classes are imbalanced: Moderate is common (43%), while Undercontroller is rare (12%).
# This is why the notebook uses macro F1 instead of accuracy.
counts = y.value_counts()

target_overview = pd.DataFrame({"count": counts,"share_%": (counts / len(y) * 100).round(1)})
target_overview.index.name = "personality type"
display(target_overview)

plt.figure(figsize=(7, 3.5))
sns.countplot(x=y, order=target_order, hue=y, palette=TARGET_COLORS, legend=False)
plt.title("How many people of each personality type?", color=THEME["primary"])
plt.xlabel("")
plt.ylabel("")
plt.show()

## 2. The preprocessing pipeline

Models need numbers, with no gaps and on a comparable scale. Our columns need different treatment:

- **numbers** (the 19 statements + `age`): fill any gaps with the median, then scale with `StandardScaler`
- **categories** (`gender`, `hand`): fill any gaps with the most frequent value, then one-hot encode, so that every category becomes its own 0/1 column

The cell below lists the columns of each group by name (2.1) and combines the two small pipelines in a `ColumnTransformer` (2.2). Every model reuses this preprocessor, so each model sees exactly the same prepared data and the comparison is fair. `handle_unknown="ignore"` turns a category the model has never seen into all zeros instead of an error, which makes the app more robust.

The target stays as **text** (`"Resilient"`), so predictions come out readable in the app, without a label encoder.

The preprocessor is only defined and displayed here, not fitted. It is fitted as part of each model's pipeline, and during cross-validation it is refitted inside every fold on the training part only, so nothing from a validation fold leaks into the scaling.

In [ ]:
# 2.1 Column definitions
question_cols = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
numeric_cols = question_cols + ['age']
categorical_cols = ['gender', 'hand']

# 2.2 Preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
]) 

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
]) 

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

display(preprocessor)

## 3. The metric and the model pipeline Setup
* **Metric** (f1_macro over Accuracy): With the Moderate class at 43%, accuracy flatters models ignoring minority classes (e.g., Undercontroller at 12%). Macro F1 weights all classes equally (`SCORING = "f1_macro"`; always-Moderate baseline macro F1 is 0.150). Accuracy is included only as a reference.

* **Pipelines**: Each model uses a dedicated pipeline combining a shared preprocessor and classifier:

    * `pipe_hgb` & `pipe_hgb_tuned`: HistGradientBoosting instances with separate grid searches (learning rate/iterations vs. tree size/class weights).

    * **CatBoost**: Fifth model; tuned via a small grid search (no Hyperopt) due to long training times. `allow_writing_files=False` suppresses the `catboost_info` log folder.

In [ ]:
# 3.1. Pipeline for Logistic Regression
pipe_logreg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# 3.2. Pipeline for KNN
pipe_knn = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", KNeighborsClassifier()),
    ]
)

# 3.3. Pipeline for Random Forest
pipe_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

# 3.4a. Pipeline for Hist Gradient Boosting
pipe_hgb = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.4b. Pipeline for Hist Gradient Boosting (Tuned variant)
pipe_hgb_tuned = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

# 3.5. Pipeline for CatBoost
pipe_cat = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", CatBoostClassifier(random_state=42, verbose=0)),
    ]
)

df_pipelines = pd.DataFrame([
    {
        "Pipeline": "<code>pipe_logreg</code>",
        "Model": "LogisticRegression",
        "Family": "Linear Model",
        "Why?": "Simple, highly interpretable baseline."
    },
    {
        "Pipeline": "<code>pipe_knn</code>",
        "Model": "KNeighborsClassifier",
        "Family": "Distance-based",
        "Why?": "Classifies based on similar answers; scaling is important."
    },
    {
        "Pipeline": "<code>pipe_rf</code>",
        "Model": "RandomForestClassifier",
        "Family": "Bagging / Decision Trees",
        "Why?": "Captures non-linear relationships and feature interactions."
    },
    {
        "Pipeline": "<code>pipe_hgb</code>",
        "Model": "HistGradientBoostingClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Histogram-based boosting as a fast candidate for tabular data."
    },
    {
        "Pipeline": "<code>pipe_hgb_tuned</code>",
        "Model": "HistGradientBoostingClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Optimized variant of HistGradientBoosting for performance gains."
    },
    {
        "Pipeline": "<code>pipe_cat</code>",
        "Model": "CatBoostClassifier",
        "Family": "Boosting / Decision Trees",
        "Why?": "Robust boosting method, strong performance on structured data."
    }
])

html_table = f"""
<table style="width:100%; border-collapse:collapse; margin:12px 0; font-size:14px; font-family:sans-serif; text-align:left;">
    <thead>
        <tr style="background:{THEME['success_bg']}; border-bottom:2px solid {THEME['success_border']};">
            <th style="padding:10px 14px; color:{THEME['primary']};">Pipeline</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Model</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Family</th>
            <th style="padding:10px 14px; color:{THEME['primary']};">Why?</th>
        </tr>
    </thead>
    <tbody>
"""

for i, row in df_pipelines.iterrows():
    bg_color = "#fcfdfd" if i % 2 == 0 else f"{THEME['success_bg']}55"
    html_table += f"""
        <tr style="background:{bg_color}; border-bottom:1px solid #eaeaea;">
            <td style="padding:10px 14px;">{row['Pipeline']}</td>
            <td style="padding:10px 14px;">{row['Model']}</td>
            <td style="padding:10px 14px;">{row['Family']}</td>
            <td style="padding:10px 14px;">{row['Why?']}</td>
        </tr>
    """

html_table += """
    </tbody>
</table>
"""

display(HTML(html_table))

display(pipe_hgb)   # show one pipeline as an example: the shared preprocessor + the classifier

## 4. Baseline comparison: five models with default settings
**Default Training**: All pipelines train with default settings to establish a performance baseline.

**Metrics**:
>
>* 5-fold cross-validation: Evaluates stability and mean macro F1 across five folds.
>* Training time: Tracked for each model run.
>* Test score: Single macro F1 evaluation on the test set for reference.
>

**Selection**: The 🏆 box highlights the model with the highest cross-validated macro F1, matching the final selection rule in section 7.

In [ ]:
# 4. Baseline comparison: every model with its default settings
base_pipelines = {
    "Logistic Regression": pipe_logreg,
    "K-Nearest Neighbors": pipe_knn,
    "Random Forest": pipe_rf,
    "HistGradientBoosting": pipe_hgb,
    "CatBoost": pipe_cat
}
base_results = []
base_results_html = ""

# Track execution times for relative progress bar calculation
durations = []
temp_runs = []
for name, pipe in tqdm(base_pipelines.items(), desc="Training Base Models"):
    start_time = time.time()
    pipe.fit(X_train, y_train)
    elapsed_time = time.time() - start_time
    
    fold_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()
    test_score = f1_score(y_test,np.ravel(pipe.predict(X_test)),average="macro",)
    
    durations.append(elapsed_time)
    temp_runs.append({
        "name": name,
        "model": pipe,
        "test_score": test_score,
        "cv_score": mean_cv,
        "elapsed_time": elapsed_time,
        "fold_scores": fold_scores,
        "std_cv": std_cv
    })

max_time = max(durations) if durations else 1

for run in temp_runs:
    name = run["name"]
    elapsed_time = run["elapsed_time"]
    mean_cv = run["cv_score"]
    std_cv = run["std_cv"]
    test_score = run["test_score"]
    fold_scores = run["fold_scores"]
    
    # Dynamic coloring based on execution time using THEME values
    time_color = THEME['success_border'] if elapsed_time < 3 else (THEME['success_border'] if elapsed_time < 10 else THEME['danger_border'])
    bar_width = int((elapsed_time / max_time) * 100)

    # Compact fold display
    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']}; border:1px solid {THEME['success_border']}; border-radius:4px; padding:4px 6px; text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700; color:{THEME['primary']};">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    base_results.append({
        "name": name,
        "model": run["model"],
        "test_score": test_score,
        "cv_score": mean_cv
    })

    base_results_html += f"""
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-bottom:10px; font-family:sans-serif;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
            <h4 style="margin:0; color:{THEME['primary']}; font-size:16px;">{name} (Default)</h4>
            <div style="display:flex; align-items:center; gap:8px; width:150px;">
                <div style="flex-grow:1; background:#e0e0e0; border-radius:3px; height:6px; overflow:hidden;">
                    <div style="width:{bar_width}%; background:{time_color}; height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;">{elapsed_time:.1f}s</span>
            </div>
        </div>
        
        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">CV-Score (Mean F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:{THEME['primary']};">{mean_cv:.3f} <span style="font-size:11px; color:#666; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">Test-Score (F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>
        
        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """

# the best model is the one with the highest cross-validated macro F1 (as in section 7)
best_base_overall = max(base_results, key=lambda x: x["cv_score"])

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px; color:{THEME['primary']};">Base Model Comparison (Without Tuning) & F1 Macro Details</h3>
    {base_results_html}
    
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px 0; color:{THEME['primary']}; font-size:16px;">🏆 Best Base Model: {best_base_overall['name']}</h4>
        <div style="font-size:12px; color:#333;">
            Achieved the highest performance with a <b>Test-Score (F1 Macro) of {best_base_overall['test_score']:.3f}</b>.
        </div>
    </div>
</div>
"""))

### What the baseline comparison shows

- **CatBoost** starts highest (macro F1 0.806), ahead of **HistGradientBoosting** (0.797). Both are boosted tree models, which tend to do best on tabular data like ours.
- **Logistic Regression** follows at 0.767. That a linear model gets this close shows that much of the signal is simple: agreeing strongly with the stress and mood statements points towards Overcontroller, disagreeing towards Resilient.
- **Random Forest** (0.741) and **K-Nearest Neighbors** (0.726) come last with their default settings.
- The five fold scores are close together for every model, so the ranking does not depend on one lucky split.
- The timing bars show the cost: CatBoost takes several times longer to train than the other models, which is why its grid in section 5 is kept small and it is left out of Hyperopt.

## 5. Hyperparameter tuning, part 1: grid search

A hyperparameter is a setting we choose before training, for example how deep a tree may grow or how strong the regularisation is. Tuning means searching for the combination that scores best.

The brief asks for at least three models, each with tuning. We tune **all five** with grid search, and the four faster ones (Logistic Regression, K-Nearest Neighbors, Random Forest and HistGradientBoosting) also with Hyperopt in section 6:

| Method | How it searches | Good for |
|---|---|---|
| **GridSearchCV** (this section) | Tries every combination in a list we write | Few settings, values we can name in advance |
| **Hyperopt (TPE)** (section 6) | Learns from the trials it already ran and samples where the score looks promising | Many settings, or continuous ranges |

Both methods score every candidate on the same five folds `cv` with macro F1, and keep the settings with the highest cross-validated score. Each tuned pipeline is also scored on the test set and shown next to it, but the champion is picked by the cross-validated score only (section 7).

> **Note:** The brief asks for tuning on at least three models, so we tune all four. Without that rule we might have skipped Random Forest and K-Nearest Neighbors, because they were furthest behind before tuning. The results below show that would have been a mistake for Random Forest: it gains the most of all four models from tuning (about +0.04 macro F1) and ends up close to Logistic Regression.

### 5.1 Grid search

`GridSearchCV` scores **every** combination in the grid, so it is exhaustive — but the cost multiplies. Two settings with 2 and 4 values is already 8 combinations × 5 folds = 40 fits.

Note the `classifier__` prefix in the parameter names. It means *"the `C` of the pipeline step called `classifier`"*. That is how we reach inside the pipeline. Since the whole pipeline goes into the search, the preprocessor is refit inside every fold — so nothing leaks from the validation fold into the scaling.

In [ ]:
# 5.1 Dictionary for GridSearch Tuning & Models Configuration
models_config = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "params": {
            "classifier__C": [0.01, 0.1, 1, 10],
            "classifier__solver": ["lbfgs"],
        },
    },
    "KNN": {
        "pipeline": pipe_knn,
        "params": {
            "classifier__n_neighbors": [3, 5, 7, 9],
            "classifier__weights": ["uniform", "distance"],
        },
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "params": {
            "classifier__n_estimators": [100, 200],
            "classifier__max_depth": [None, 10, 20],
            "classifier__min_samples_split": [2, 5],
        },
    },
    "HistGradient Boosting": {
        "pipeline": pipe_hgb,
        "params": {
            "classifier__learning_rate": [0.01, 0.05, 0.1],
            "classifier__max_iter": [100, 200],
        },
    },
    "HistGradient Boosting Tuned": {
        "pipeline": pipe_hgb_tuned,
        "params": {
            "classifier__max_leaf_nodes": [31, 32],
            "classifier__min_samples_leaf": [20, 60],
            "classifier__class_weight": ["balanced"],
        },
    },
    # A large CatBoost grid (iterations x depth x learning_rate x l2_leaf_reg = 81 combinations)
    # was too slow, so CatBoost gets a small grid: tree depth and class weights.
    # Two parts, because CatBoost does not accept auto_class_weights=None:
    # the first part trains without class weights, the second part with them.
    "CatBoost": {
        "pipeline": pipe_cat,
        "params": [
            {"classifier__depth": [4, 6]},
            {"classifier__depth": [4, 6],
             "classifier__auto_class_weights": ["SqrtBalanced", "Balanced"]},
        ],
    },
}

In [ ]:
# 5.2 Run the grid search for every model
results = []
results_html = ""

# Track execution times for relative progress bar calculation
durations_grid = []
temp_grid_runs = []

for name, config in tqdm(models_config.items(), desc="Grid Search Tuning"):
    start_time = time.time()

    grid_search = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring={
            "f1_macro": "f1_macro",
            "accuracy": "accuracy"
        },
        refit="f1_macro",
        return_train_score=True,
        n_jobs=-1,
        error_score="raise"
    )

    grid_search.fit(X_train, y_train)
    elapsed_time = time.time() - start_time

    best_model = grid_search.best_estimator_
    best_index = grid_search.best_index_
    search_scores = grid_search.cv_results_

    mean_cv = search_scores["mean_test_f1_macro"][best_index]
    std_cv = search_scores["std_test_f1_macro"][best_index]
    # the five fold scores of the best combination, taken from the search (no need to train again)
    fold_scores = [search_scores[f"split{k}_test_f1_macro"][best_index] for k in range(N_SPLITS)]
    # macro F1 on the test set (np.ravel: CatBoost returns its predictions as a column)
    test_score = f1_score(y_test, np.ravel(best_model.predict(X_test)), average="macro")

    durations_grid.append(elapsed_time)
    temp_grid_runs.append({
        "name": name,
        "model": best_model,
        "mean_cv": mean_cv,
        "std_cv": std_cv,
        "test_score": test_score,
        "elapsed_time": elapsed_time,
        "best_params": grid_search.best_params_,
        "fold_scores": fold_scores
    })

max_grid_time = max(durations_grid) if durations_grid else 1

for run in temp_grid_runs:
    name = run["name"]
    elapsed_time = run["elapsed_time"]
    mean_cv = run["mean_cv"]
    std_cv = run["std_cv"]
    test_score = run["test_score"]
    best_params = run["best_params"]
    fold_scores = run["fold_scores"]

    # Colour the run-time bar by duration (THEME colours)
    time_color = THEME['success_border'] if elapsed_time < 15 else (THEME['success_border'] if elapsed_time < 45 else THEME['danger_border'])
    bar_width = int((elapsed_time / max_grid_time) * 100)

    # Show the best parameters in a readable way (without the 'classifier__' prefix)
    best_params_str = ", ".join([f"<b>{k.replace('classifier__', '')}</b>: {v}" for k, v in best_params.items()])

    results.append({
        "name": name,
        "model": run["model"],
        "test_score": test_score,
        "cv_score": mean_cv
    })

    # Compact display of the five fold scores in the theme colours
    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']}; border:1px solid {THEME['success_border']}; border-radius:4px; padding:4px 6px; text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700; color:{THEME['primary']};">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    results_html += f"""
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-bottom:10px; font-family:sans-serif;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
            <h4 style="margin:0; color:{THEME['primary']}; font-size:16px;">{name} (Grid Search)</h4>
            <div style="display:flex; align-items:center; gap:8px; width:150px;">
                <div style="flex-grow:1; background:#e0e0e0; border-radius:3px; height:6px; overflow:hidden;">
                    <div style="width:{bar_width}%; background:{time_color}; height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;">{elapsed_time:.1f}s</span>
            </div>
        </div>

        <div style="font-size:12px; color:#444; margin-bottom:8px;">
            <b>Best Parameters:</b> {best_params_str}
        </div>

        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">CV-Score (Mean F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:{THEME['primary']};">{mean_cv:.3f} <span style="font-size:11px; color:#666; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">Test-Score (F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>

        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """

# the best grid-search run is the one with the highest cross-validated macro F1
best_overall = max(results, key=lambda x: x["cv_score"])

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px; color:{THEME['primary']};">Model Comparison, Tuning & F1 Macro Details</h3>
    {results_html}

    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px 0; color:{THEME['primary']}; font-size:16px;">🏆 Best Grid Search Model: {best_overall['name']}</h4>
        <div style="font-size:12px; color:#333;">
            Highest cross-validated macro F1: <b>{best_overall['cv_score']:.3f}</b> (test macro F1: {best_overall['test_score']:.3f}).
        </div>
    </div>
</div>
"""))

## 6. Hyperparameter tuning, part 2: Hyperopt

### 6.1 How Hyperopt searches

The same models except CatBoost, but now with **ranges** instead of lists. The learning rate of HistGradientBoosting, for example, can be anything between 0.02 and 0.3, not just the three values the grid could afford. Hyperopt tries a setting, sees the score, and uses that to decide what to try next (the TPE algorithm), so a few trials can cover a wide space. It runs `MAX_EVALS = 10` trials per model, each scored on the same five folds `cv` with macro F1.

Three things to know about the syntax:

- `hp.loguniform` samples across orders of magnitude (0.01, 0.1, 1 and 10 are equally likely), which suits `C`
- `hp.quniform(low, high, step)` samples in steps, and `scope.int(...)` turns the result into a whole number, because settings like `n_estimators` must be integers
- Hyperopt **minimises**, so the objective returns `-score`

In [ ]:
# 6.2 Dictionary for Hyperopt Tuning & Models Configuration and Spaces
hyperopt_configs = {
    "Logistic Regression": {
        "pipeline": pipe_logreg,
        "space": {
            "classifier__C": hp.loguniform("classifier__C", np.log(0.01), np.log(10))
        }
    },
    "K-Nearest Neighbors": {
        "pipeline": pipe_knn,
        "space": {
            # every whole number from 3 to 11 (a step of 2 would only give 4, 6, 8 and 10)
            "classifier__n_neighbors": scope.int(hp.quniform("classifier__n_neighbors", 3, 11, 1))
        }
    },
    "Random Forest": {
        "pipeline": pipe_rf,
        "space": {
            "classifier__n_estimators": scope.int(hp.quniform("classifier__n_estimators", 50, 300, 50)),
            "classifier__max_depth": scope.int(hp.quniform("classifier__max_depth", 5, 30, 5)),
            "classifier__min_samples_split": scope.int(hp.quniform("classifier__min_samples_split", 2, 10, 2))
        }
    },
    "HistGradientBoosting": {
        "pipeline": pipe_hgb,
        "space": {
            "classifier__learning_rate": hp.uniform("classifier__learning_rate", 0.02, 0.3),
            "classifier__max_iter": scope.int(hp.quniform("classifier__max_iter", 50, 250, 50))
        }
    }
}


In [ ]:
# 6.3 Run Hyperopt for every model
hyperopt_results = []
hyperopt_results_html = ""
durations_hyper = []
temp_hyper_runs = []

for name, config in tqdm(hyperopt_configs.items(), desc="Hyperopt Tuning"):
    start_time = time.time()

    # a fresh copy, so the untuned pipeline from section 4 stays unchanged
    pipeline = clone(config["pipeline"])
    space = config["space"]

    def objective(params):
        pipeline.set_params(**params)
        scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
        return {'loss': -scores.mean(), 'status': STATUS_OK}

    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=MAX_EVALS,
        trials=trials,
        rstate=np.random.default_rng(RANDOM_STATE),
        verbose=0
    )

    elapsed_time = time.time() - start_time

    # Hyperopt returns these settings as floats, but they must be whole numbers
    best_params = {}
    for k, v in best.items():
        if "n_estimators" in k or "max_depth" in k or "min_samples_split" in k or "n_neighbors" in k or "max_iter" in k:
            best_params[k] = int(v)
        else:
            best_params[k] = v

    # Set the best settings on the pipeline and train it on the full training set
    pipeline.set_params(**best_params)
    pipeline.fit(X_train, y_train)

    fold_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1)
    mean_cv = fold_scores.mean()
    std_cv = fold_scores.std()
    # macro F1 on the test set
    test_score = f1_score(y_test, np.ravel(pipeline.predict(X_test)), average="macro")

    durations_hyper.append(elapsed_time)
    temp_hyper_runs.append({
        "name": name,
        "model": pipeline,
        "mean_cv": mean_cv,
        "std_cv": std_cv,
        "test_score": test_score,
        "elapsed_time": elapsed_time,
        "best_params": best_params,
        "fold_scores": fold_scores
    })

max_hyper_time = max(durations_hyper) if durations_hyper else 1

for run in temp_hyper_runs:
    name = run["name"]
    elapsed_time = run["elapsed_time"]
    mean_cv = run["mean_cv"]
    std_cv = run["std_cv"]
    test_score = run["test_score"]
    best_params = run["best_params"]
    fold_scores = run["fold_scores"]

    # Colour the run-time bar by duration (THEME colours)
    time_color = THEME['success_border'] if elapsed_time < 15 else (THEME['success_border'] if elapsed_time < 45 else THEME['danger_border'])
    bar_width = int((elapsed_time / max_hyper_time) * 100)

    # Show the best parameters in a readable way (without the 'classifier__' prefix, floats rounded)
    best_params_str = ", ".join([f"<b>{k.replace('classifier__', '')}</b>: {v:.4g}" if isinstance(v, float) else f"<b>{k.replace('classifier__', '')}</b>: {v}" for k, v in best_params.items()])

    hyperopt_results.append({
        "name": name,
        "model": run["model"],
        "test_score": test_score,
        "cv_score": mean_cv
    })

    # Compact display of the five fold scores in the theme colours
    folds_html = "".join(
        f"""
        <div style="background:{THEME['success_bg']}; border:1px solid {THEME['success_border']}; border-radius:4px; padding:4px 6px; text-align:center; flex:1;">
            <div style="color:#555; font-size:10px;">F{i}</div>
            <div style="font-size:13px; font-weight:700; color:{THEME['primary']};">{score:.3f}</div>
        </div>
        """
        for i, score in enumerate(fold_scores, start=1)
    )

    hyperopt_results_html += f"""
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-bottom:10px; font-family:sans-serif;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
            <h4 style="margin:0; color:{THEME['primary']}; font-size:16px;">{name} (Hyperopt)</h4>
            <div style="display:flex; align-items:center; gap:8px; width:150px;">
                <div style="flex-grow:1; background:#e0e0e0; border-radius:3px; height:6px; overflow:hidden;">
                    <div style="width:{bar_width}%; background:{time_color}; height:100%;"></div>
                </div>
                <span style="font-size:11px; color:#555;">{elapsed_time:.1f}s</span>
            </div>
        </div>

        <div style="font-size:12px; color:#444; margin-bottom:8px;">
            <b>Best Parameters:</b> {best_params_str}
        </div>

        <div style="display:flex; gap:10px; margin-bottom:8px;">
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">CV-Score (Mean F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:{THEME['primary']};">{mean_cv:.3f} <span style="font-size:11px; color:#666; font-weight:normal;">(±{std_cv:.3f})</span></div>
            </div>
            <div style="background:#ffffff; border:1px solid #d1e7dd; border-radius:6px; padding:6px 10px; flex:1;">
                <div style="color:#555; font-size:10px;">Test-Score (F1 Macro)</div>
                <div style="font-size:15px; font-weight:700; color:#15803d;">{test_score:.3f}</div>
            </div>
        </div>

        <div style="display:flex; gap:4px;">
            {folds_html}
        </div>
    </div>
    """

# the best Hyperopt run is the one with the highest cross-validated macro F1
best_hyper_overall = max(hyperopt_results, key=lambda x: x["cv_score"])

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; margin:16px 0;">
    <h3 style="margin-bottom:12px; color:{THEME['primary']};">Hyperopt Tuning Results & Performance Details</h3>
    {hyperopt_results_html}

    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']}; border-radius:6px; padding:12px; margin-top:12px;">
        <h4 style="margin:0 0 4px 0; color:{THEME['primary']}; font-size:16px;">🏆 Best Hyperopt Model: {best_hyper_overall['name']}</h4>
        <div style="font-size:12px; color:#333;">
            Highest cross-validated macro F1: <b>{best_hyper_overall['cv_score']:.3f}</b> (test macro F1: {best_hyper_overall['test_score']:.3f}).
        </div>
    </div>
</div>
"""))

### What tuning changed

- **CatBoost** gains the most: with mild class weights (`auto_class_weights="SqrtBalanced"`) and shallower trees (`depth=4`) its cross-validated macro F1 rises from 0.806 to **0.819**, the best score of the notebook.
- **HistGradientBoosting** improves slightly with either method (0.797 to 0.798). Its balanced grid reaches 0.803.
- **K-Nearest Neighbors** improves with both methods, from 0.726 to 0.739 with the grid search (9 neighbours, weighted by distance) and to 0.731 with Hyperopt (9 neighbours), but it stays the weakest model.
- **Logistic Regression** and **Random Forest** were already at their best with the default settings: tuning leaves their scores practically unchanged.
- Grid search and Hyperopt end up at practically the same scores, except for K-Nearest Neighbors: the grid also tries `weights="distance"`, which the Hyperopt search space doesn't include. With ten trials, Hyperopt does not find anything the grids missed.

## 7. Results: the comparison and the winner

All runs from sections 4 to 6 go into one table: five untuned models, six grid searches and four Hyperopt searches, 15 runs in total.

The **champion is the run with the highest cross-validated macro F1**, the metric we committed to in section 3. The test score is reported alongside it as an independent check, not as the selection criterion (choosing by the test score would spend the one honest number we have).

### 7.1 The champion

The cell picks the champion, trains a fresh copy of it on the training set (`clone` + `fit`), and scores the always-Moderate baseline from section 3 for comparison.

In [ ]:
# 7.1 Combine all runs and select the champion by CV macro F1

def result_rows(runs, tuning):
    rows = []
    for run in runs:
        y_pred = np.ravel(run["model"].predict(X_test))   # np.ravel: CatBoost returns a column
        rows.append({
            "model": run["name"],
            "tuning": tuning,
            "pipeline": run["model"],
            "cv_f1_macro": run["cv_score"],
            "test_f1_macro": f1_score(y_test, y_pred, average="macro"),
            "test_accuracy": accuracy_score(y_test, y_pred),
            # untuned runs use the default settings, so there are no tuned parameters to show
            "params": {} if tuning == "none" else run.get("best_params", {}),
        })
    return rows

# Use the tuned-run lists directly so the summary stays correct when searches are changed.
grid_runs = [
    {**run, "cv_score": run["mean_cv"]}
    for run in temp_grid_runs
]
hyperopt_runs = [
    {**run, "cv_score": run["mean_cv"]}
    for run in temp_hyper_runs
]
all_runs = pd.DataFrame(
    result_rows(temp_runs, "none")
    + result_rows(grid_runs, "grid search")
    + result_rows(hyperopt_runs, "hyperopt")
)

champion_row = all_runs.loc[all_runs["cv_f1_macro"].idxmax()]
CHAMPION_NAME = champion_row["model"]
CHAMPION_TUNING = champion_row["tuning"]
CHAMPION_CV = champion_row["cv_f1_macro"]
CHAMPION_TEST = champion_row["test_f1_macro"]
champion_test_acc = champion_row["test_accuracy"]
champion_params = champion_row["params"]
champion_pipeline = clone(champion_row["pipeline"])
champion_pipeline.fit(X_train, y_train)

# A simple, class-frequency-aware reference point.
majority_class = y_train.mode().iat[0]
baseline_pred = np.full(shape=len(y_test), fill_value=majority_class, dtype=object)
baseline_f1 = f1_score(y_test, baseline_pred, average="macro")

def format_params(params):
    if not params:
        return "the default parameters"
    return ", ".join(
        f"{key.replace('classifier__', '')}={value}" for key, value in params.items()
    )

why_selected = (
    f"It achieved the highest mean cross-validated macro F1 ({CHAMPION_CV:.3f}) "
    f"across all evaluated runs. Its held-out test macro F1 was {CHAMPION_TEST:.3f}."
)

display(HTML(f"""
<div style="font-family:sans-serif; max-width:850px; background:{THEME['success_bg']}; border-left:5px solid {THEME['success_border']}; border-radius:6px; padding:16px; margin:12px 0;">
  <div style="font-size:13px; color:#555; margin-bottom:5px;">Selected by cross-validated macro F1</div>
  <h2 style="margin:0 0 8px; color:{THEME['primary']};">Champion: {CHAMPION_NAME}</h2>
  <p style="margin:0 0 10px;">{why_selected}</p>
  <table style="border-collapse:collapse; width:100%; background:#fff;">
    <tr><th style="text-align:left; padding:7px;">Search method</th><td style="padding:7px;">{CHAMPION_TUNING.title()}</td></tr>
    <tr><th style="text-align:left; padding:7px;">CV macro F1</th><td style="padding:7px;"><b>{CHAMPION_CV:.3f}</b></td></tr>
    <tr><th style="text-align:left; padding:7px;">Test macro F1</th><td style="padding:7px;">{CHAMPION_TEST:.3f}</td></tr>
    <tr><th style="text-align:left; padding:7px;">Test accuracy</th><td style="padding:7px;">{champion_test_acc:.1%}</td></tr>
    <tr><th style="text-align:left; padding:7px;">Best parameters</th><td style="padding:7px;">{format_params(champion_params)}</td></tr>
  </table>
</div>
"""))


### 7.2 Confusion matrix of the champion on the test set

Each row is a true class and shows what share of it the champion predicted as which class. The diagonal is therefore the recall of each class: the share of that type the model finds.

In [ ]:
# 7.2 Confusion matrix for the selected model
# Each row represents a true class. Percentages are row-normalised;
# values in parentheses are absolute counts.

y_pred_test = np.ravel(champion_pipeline.predict(X_test))

cm = confusion_matrix(
    y_test,
    y_pred_test,
    labels=target_order,
    normalize="true",
)

cm_counts = confusion_matrix(
    y_test,
    y_pred_test,
    labels=target_order,
)

annotations = np.array([
    [
        f"{percentage:.1%}\n({count})"
        for percentage, count in zip(percent_row, count_row)
    ]
    for percent_row, count_row in zip(cm, cm_counts)
])

fig, ax = plt.subplots(figsize=(8, 5.7))

sns.heatmap(
    cm,
    annot=annotations,
    fmt="",
    cmap=THEME["palette"],
    xticklabels=target_order,
    yticklabels=target_order,
    vmin=0,
    vmax=1,
    cbar_kws={"label": "Share of true class"},
    ax=ax,
)

ax.set_title(
    f"Confusion matrix: {CHAMPION_NAME} ({CHAMPION_TUNING})",
    color=THEME["primary"],
)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")

fig.text(
    0.5,
    0.01,
    "Rows are true classes. Percentages are row-normalised; values in parentheses are absolute counts.",
    ha="center",
    fontsize=10,
    color="#555555",
)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

In [ ]:
# 7.3 Full leaderboard
leaderboard = all_runs[["model", "tuning", "cv_f1_macro", "test_f1_macro", "test_accuracy"]].copy()
leaderboard = leaderboard.sort_values(["cv_f1_macro", "test_f1_macro"], ascending=False).reset_index(drop=True)
leaderboard.index = leaderboard.index + 1
leaderboard.index.name = "rank"
leaderboard["tuning"] = leaderboard["tuning"].replace({"none": "untuned"})

display(
    leaderboard.style
    .format({"cv_f1_macro": "{:.3f}", "test_f1_macro": "{:.3f}", "test_accuracy": "{:.1%}"})
    .highlight_max(subset=["cv_f1_macro"], color=THEME["success_bg"])
    .set_caption("Runs are ranked by cross-validated macro F1; test results are shown only as a final check.")
)


## 8. The champion on the test set

The champion was chosen with cross-validation on the training data. Now we look at how it
behaves on the 20% it has never seen — per class, which is where the interesting part is.

In [ ]:
# 8. Detailed evaluation on the held-out test set
report = classification_report(
    y_test,
    y_pred_test,
    labels=target_order,
    target_names=target_order,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T.loc[
    target_order + ["macro avg", "weighted avg"]
]

test_macro_f1 = f1_score(y_test, y_pred_test, average="macro")
test_accuracy = accuracy_score(y_test, y_pred_test)

# Summary cards
display(HTML(f"""
<div style="display:flex; gap:12px; margin:12px 0; font-family:sans-serif;">
    <div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']};
                border-radius:6px; padding:12px 16px; min-width:210px;">
        <div style="font-size:12px; color:#555;">Held-out test macro F1</div>
        <div style="font-size:24px; font-weight:700; color:{THEME['primary']};">
            {test_macro_f1:.3f}
        </div>
    </div>
    <div style="background:#f7f9fa; border-left:4px solid {THEME['primary']};
                border-radius:6px; padding:12px 16px; min-width:210px;">
        <div style="font-size:12px; color:#555;">Held-out test accuracy</div>
        <div style="font-size:24px; font-weight:700; color:{THEME['primary']};">
            {test_accuracy:.1%}
        </div>
    </div>
</div>
"""))

# Classification report
report_display = report_df[["precision", "recall", "f1-score", "support"]].copy()
report_display.index.name = "Personality type"

display(
    report_display.style
    .format({
        "precision": "{:.3f}",
        "recall": "{:.3f}",
        "f1-score": "{:.3f}",
        "support": "{:.0f}",
    })
    .background_gradient(
        subset=["precision", "recall", "f1-score"],
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    .set_caption("Classification report on the held-out test set")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "16px"),
                ("font-weight", "700"),
                ("color", THEME["primary"]),
                ("text-align", "left"),
                ("padding-bottom", "8px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", THEME["success_bg"]),
                ("color", THEME["primary"]),
                ("font-weight", "700"),
                ("text-align", "left"),
                ("padding", "8px 12px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "8px 12px"),
                ("text-align", "center"),
            ],
        },
    ])
)

### What the test set shows

- The champion performs similarly on unseen data (test macro F1: **0.823**; CV macro F1: **0.819**).
- **Resilient**, **Overcontroller**, and **Moderate** are classified well.
- **Undercontroller** remains difficult: recall is **0.575**, and most missed cases are predicted as Moderate.

Only `C4` and `A4` measure the two traits that define Undercontroller, so the reduced 19-item questionnaire provides limited information for this class.

`SqrtBalanced` class weights improve Undercontroller recall from 44% to 58% with almost no loss in accuracy. Fully balanced weights find more Undercontrollers, but reduce overall accuracy and macro F1.

## 9. Fit the champion on all the data and save it with joblib

Cross-validation on the training set showed which pipeline performs best, and the held-out test set confirmed that result. Since the test score has already been measured, the test rows aren't needed for evaluation anymore. So for the final model we refit a fresh copy of the champion pipeline on the full dataset (`X` and `y`: training and test data combined), so it learns from every row we have before it goes into `app.py`.

We save the **complete pipeline** with `joblib`, not just the classifier. The pipeline contains the preprocessing steps (imputation, scaling and one-hot encoding) and the tuned classifier. When the app passes in a raw row of user answers, the pipeline transforms it exactly the same way as the training data and then makes the prediction.

The fitted pipeline is called `best_pipeline`, as in the project brief, and saved as `models/personality_pipeline.joblib`, the file name from the brief and the file `app.py` loads. The cell creates the `models/` folder if it doesn't exist yet.

> The `models/` folder is listed in `.gitignore`, so it isn't committed to the repository. Running this notebook creates the model file again. Every step that involves randomness uses `random_state=42`, so you get the same model every time.

In [ ]:
# 9.1 Refit the selected pipeline on all available data and save it
best_pipeline = clone(champion_pipeline)
best_pipeline.fit(X, y)

model_path = Path("models") / "personality_pipeline.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, model_path)

display(HTML(f"""
<div style="font-family:sans-serif; max-width:700px; background:{THEME['success_bg']};
            border-left:4px solid {THEME['success_border']}; border-radius:6px;
            padding:14px 16px; margin:12px 0;">
    <div style="font-size:18px; font-weight:700; color:{THEME['primary']}; margin-bottom:8px;">
        Pipeline saved successfully
    </div>
    <table style="border-collapse:collapse; font-size:14px;">
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">File</td>
            <td style="padding:3px 0;"><code>{model_path}</code></td>
        </tr>
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">Model</td>
            <td style="padding:3px 0;"><b>{CHAMPION_NAME}</b> ({CHAMPION_TUNING})</td>
        </tr>
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">Training rows</td>
            <td style="padding:3px 0;">{len(X):,}</td>
        </tr>
        <tr>
            <td style="padding:3px 16px 3px 0; color:#555;">Classes</td>
            <td style="padding:3px 0;">{", ".join(best_pipeline.classes_)}</td>
        </tr>
    </table>
</div>
"""))

## 10. From the notebook to the app

`app.py` loads the saved pipeline from `models/personality_pipeline.joblib`; it does not train a model.

The cells below recreate the app workflow:

1. **10.1** Reloads the pipeline and checks that a sample input has the same columns, order, and data types as the training data.
2. **10.2** Predicts a personality type from that raw input.

`np.ravel(...)[0]` returns a single class label and also handles CatBoost’s nested prediction format.

Start the app with:

```bash
streamlit run app.py

## 11. Final decision

The closing paragraph the project brief asks for. The numbers are filled in from the results
above, so the text always matches the latest run, even if the cleaned data changes.

In [ ]:
# 10.1 Load the saved pipeline again, like app.py does at startup
loaded_pipeline = joblib.load(model_path)

# One made-up user: a calm, sociable person
# Keys = column names from the codebook, in the same order as the training data
# Scale for the 19 statements: 1 = Disagree, 2 = Slightly Disagree, 3 = Neutral, 4 = Slightly Agree, 5 = Agree
user_answers = {
    "N6":  1,            # I get upset easily.
    "N8":  1,            # I have frequent mood swings.
    "N7":  2,            # I change my mood a lot.
    "N1":  2,            # I get stressed out easily.
    "N9":  2,            # I get irritated easily.
    "N10": 1,            # I often feel blue.
    "N3":  2,            # I worry about things.
    "N5":  1,            # I am easily disturbed.
    "E3":  5,            # I feel comfortable around people.
    "N2":  4,            # I am relaxed most of the time.
    "E5":  4,            # I start conversations.
    "N4":  4,            # I seldom feel blue.
    "E7":  4,            # I talk to a lot of different people at parties.
    "E4":  2,            # I keep in the background.
    "age": 29,           # age in years
    "C4":  2,            # I make a mess of things.
    "E1":  3,            # I am the life of the party.
    "A4":  5,            # I sympathize with others' feelings.
    "E10": 2,            # I am quiet around strangers.
    "E9":  4,            # I don't mind being the center of attention.
    "gender": "Female",  # 'Male', 'Female' or 'Other'
    "hand":   "Right",   # 'Right', 'Left' or 'Both'
}

# 10.2 Predict from one raw user input row
input_df = pd.DataFrame([user_answers])

columns_match = list(input_df.columns) == list(X.columns)
dtypes_match = input_df.dtypes.equals(X.dtypes)

# The complete pipeline applies preprocessing before making the prediction.
prediction = np.ravel(loaded_pipeline.predict(input_df))[0]
probabilities = loaded_pipeline.predict_proba(input_df)[0]

check_colour = THEME["success_border"] if columns_match and dtypes_match else THEME["danger_border"]
check_message = "Input structure matches the training data." if columns_match and dtypes_match else "Input structure does not match the training data."

display(HTML(f"""
<div style="font-family:sans-serif; max-width:680px;">
    <div style="background:{THEME['success_bg']}; border-left:4px solid {check_colour};
                border-radius:6px; padding:12px 16px; margin-bottom:12px;">
        <b>{check_message}</b><br>
        <span style="font-size:13px;">
            Same column names and order: <b>{columns_match}</b><br>
            Same data types: <b>{dtypes_match}</b>
        </span>
    </div>

    <div style="background:#f7f9fa; border-left:4px solid {THEME['primary']};
                border-radius:6px; padding:14px 16px;">
        <div style="font-size:12px; color:#555;">Predicted personality type</div>
        <div style="font-size:24px; font-weight:700; color:{THEME['primary']};">
            {prediction}
        </div>
    </div>
</div>
"""))

probability_df = pd.DataFrame({
    "Personality type": loaded_pipeline.classes_,
    "Probability": probabilities,
}).sort_values("Probability", ascending=False)

display(
    probability_df.style
    .format({"Probability": "{:.1%}"})
    .bar(subset=["Probability"], color=THEME["success_border"], vmin=0, vmax=1)
    .set_caption("Prediction probabilities")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "16px"),
                ("font-weight", "700"),
                ("color", THEME["primary"]),
                ("text-align", "left"),
                ("padding-bottom", "8px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", THEME["success_bg"]),
                ("color", THEME["primary"]),
                ("text-align", "left"),
                ("padding", "8px 12px"),
            ],
        },
        {
            "selector": "td",
            "props": [("padding", "8px 12px")],
        },
    ])
)

> Both checks print `True`, and the pipeline predicts a type straight from the raw row.
> The made-up user is calm and sociable, and comes out as **Resilient**, as expected.
> The saved file is ready for the app.

## 11. Final decision

The closing paragraph the project brief asks for. The numbers in the cell below are filled in from the results above, so they always match the latest run, even if the cleaned data changes. The decision is written out in words underneath.

In [ ]:
# 11. Final decision
untuned = all_runs[all_runs["tuning"] == "none"].sort_values("cv_f1_macro", ascending=False)
matching_untuned = untuned[untuned["model"] == CHAMPION_NAME]
untuned_champion = matching_untuned.iloc[0] if not matching_untuned.empty else None

# The gain from tuning only makes sense if the champion is a tuned run with an untuned version
gain = None
if CHAMPION_TUNING != "none" and untuned_champion is not None:
    gain = CHAMPION_CV - untuned_champion["cv_f1_macro"]

tuning_text = "with its default settings" if CHAMPION_TUNING == "none" else f"tuned with {CHAMPION_TUNING}"
gain_text = (
    f"Compared with the untuned version of the same model, the cross-validated macro F1 changed by **{gain:+.3f}**. "
    if gain is not None else ""
)

model_rows = "\n".join(
    f"| **{row.model}** | **{row.cv_f1_macro:.3f}** |" if row.model == CHAMPION_NAME
    else f"| {row.model} | {row.cv_f1_macro:.3f} |"
    for row in untuned.itertuples()
)

final_decision = f"""
**Baseline comparison.** The default pipelines were evaluated with {N_SPLITS}-fold stratified cross-validation on the training set.

| Model | CV macro F1 (untuned) |
|---|---|
{model_rows}

**Final model.** {CHAMPION_NAME}, {tuning_text}, was selected because it achieved the highest cross-validated macro F1: **{CHAMPION_CV:.3f}**. Its macro F1 on the held-out test set was **{CHAMPION_TEST:.3f}**, with an accuracy of **{champion_test_acc:.1%}**. The majority-class baseline reached only **{baseline_f1:.3f}** macro F1.

{gain_text}The selected parameters were: {format_params(champion_params)}.

The similar cross-validation and held-out test scores indicate that the selected pipeline generalizes well to unseen data. This complete preprocessing-and-classification pipeline was then refitted on the full dataset and saved as `models/personality_pipeline.joblib` for the Streamlit app (section 9).
"""

display(Markdown(final_decision))

### The decision in words

Five baseline models were compared using 5-fold stratified cross-validation and `f1_macro`. CatBoost performed best initially (0.806), followed by HistGradientBoosting (0.797), Logistic Regression (0.767), Random Forest (0.741), and K-Nearest Neighbors (0.726).

All five models were tuned with `GridSearchCV`; Logistic Regression, K-Nearest Neighbors, Random Forest, and HistGradientBoosting were also tuned with Hyperopt. The searches covered `C`; `n_neighbors` and `weights`; `n_estimators`, `max_depth`, and `min_samples_split`; `learning_rate`, `max_iter`, `max_leaf_nodes`, `min_samples_leaf`, and `class_weight`; and CatBoost’s `depth` and `auto_class_weights`.

The best configuration was CatBoost with `depth=4` and `auto_class_weights="SqrtBalanced"`. It achieved the highest cross-validated `f1_macro` of **0.819** across all 15 runs, improving on its baseline score by 0.013. On the held-out test set, it reached a macro F1 of **0.823** and an accuracy of **86.9%**; the majority-class baseline achieved only 0.150 macro F1. The class weighting increased Undercontroller recall from 0.44 to 0.58 with almost unchanged accuracy. Because the cross-validation and test scores are close, the model generalises well. The complete CatBoost pipeline was refitted on all 19,588 rows and saved as `models/personality_pipeline.joblib` for the Streamlit app.


In [ ]:
total_seconds = time.perf_counter() - RUN_ALL_START
minutes, seconds = divmod(int(total_seconds), 60)

display(HTML(f"""
<div style="background:{THEME['success_bg']}; border-left:4px solid {THEME['success_border']};
            border-radius:6px; padding:12px 16px; margin-top:16px; font-family:sans-serif;">
    <b>Notebook runtime</b><br>
    The complete “Run All” process took {minutes} min {seconds} sec.
</div>
"""))